In [1]:
import os

from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/Agentic_AI.txt",encoding="utf-8")
documents = loader.load()
print(len(documents))
print(documents[0].page_content[:500])

C:\Users\hp\AppData\Local\Temp\ipykernel_10620\2780167346.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


1
Artificial intelligence (AI) is the field of creating computer systems that perform tasks commonly associated with human abilities, such as understanding language, recognizing images, making predictions, and solving problems. Machine learning is an approach to AI in which models learn patterns from data. Large language models (LLMs) are machine-learning models that can understand and generate text, but their answers may be inaccurate and should be checked when correctness matters.

Agentic AI de


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")
print(chunks[0].page_content)


Total chunks: 18
Artificial intelligence (AI) is the field of creating computer systems that perform tasks commonly


In [4]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embed = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks,embed)

In [5]:
retriever = vectorstore.as_retriever( search_type="similarity",
    search_kwargs={"k": 3})

In [6]:
query = "What is AI?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
Artificial intelligence (AI) is the field of creating computer systems that perform tasks commonly

--- Result 2 ---
recognizing images, making predictions, and solving problems. Machine learning is an approach to AI

--- Result 3 ---
Agentic AI describes AI systems that work toward a goal by selecting and carrying out actions


In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

template = """Answer the question using only the context below.
If the context does not contain enough information, say you don't know based on the context.
Do not make up facts. Be concise.

Context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
answer = chain.invoke(query)
print(answer)

AI is the field of creating computer systems that perform tasks commonly associated with human intelligence, such as recognizing images, making predictions, and solving problems.
